In [1]:
import numpy as np
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import napari
import trimesh
import pyvista as pv
import pyacvd

In [ ]:
fix_mesh = trimesh.load("/home/tmurakami/src/surface_mesh/human_tissue_morphpaper/220715_prefrontal_q2_R01_pia.ply")
mov_mesh = trimesh.load("/home/tmurakami/src/surface_mesh/human_tissue_morphpaper/220715_prefrontal_q2_R01_wm.ply")

### Fix the z resolution problem
fix_mesh.vertices = fix_mesh.vertices * np.asarray([1.5,1.0,1.0])
mov_mesh.vertices = mov_mesh.vertices * np.asarray([1.5,1.0,1.0])
target_area = 10000 # 100 micron x 100 micron
n_target_mov = int((mov_mesh.area // target_area) // 2) # division by two to convert number of faces to number of verts
n_target_fix = int((fix_mesh.area // target_area) // 2) # division by two to convert number of faces to number of verts

In [ ]:
# --- trimesh -> pyvista ---
V, F = mov_mesh.vertices, mov_mesh.faces
faces_pv = np.hstack([np.full((len(F), 1), 3, dtype=np.int64), F]).ravel()
pmesh = pv.PolyData(V, faces_pv)

# --- uniform remeshing ---
clus = pyacvd.Clustering(pmesh)
clus.subdivide(3)                 # densify first so clustering has points to work with
clus.cluster(n_target_mov)
remesh = clus.create_mesh()

# --- pyvista -> trimesh ---
faces_tm = remesh.faces.reshape(-1, 4)[:, 1:]   # drop the leading "3" per face
mov_mesh = trimesh.Trimesh(remesh.points, faces_tm, process=False)


# --- trimesh -> pyvista ---
V, F = fix_mesh.vertices, fix_mesh.faces
faces_pv = np.hstack([np.full((len(F), 1), 3, dtype=np.int64), F]).ravel()
pmesh = pv.PolyData(V, faces_pv)

# --- uniform remeshing ---
clus = pyacvd.Clustering(pmesh)
clus.subdivide(3)                 # densify first so clustering has points to work with
clus.cluster(n_target_fix)
remesh = clus.create_mesh()

# --- pyvista -> trimesh ---
faces_tm = remesh.faces.reshape(-1, 4)[:, 1:]   # drop the leading "3" per face
fix_mesh = trimesh.Trimesh(remesh.points, faces_tm, process=False)

In [4]:
fix_vertices = fix_mesh.vertices
mov_vertices = mov_mesh.vertices
fix_face = fix_mesh.faces
mov_face = mov_mesh.faces

fix_vertex_colors = np.tile(
    np.array([0.0, 1.0, 0.0, 1.0]),  # RGBA: red
    (fix_vertices.shape[0], 1)
)
mov_vertex_colors = np.tile(
    np.array([1.0, 0.0, 1.0, 1.0]),  # RGBA: red
    (mov_vertices.shape[0], 1)
)

In [5]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_points(fix_vertices, size=50, face_color='lime', blending="additive")
viewer.add_points(mov_vertices, size=50, face_color='magenta', blending="additive")
viewer.add_surface((fix_vertices,fix_face),blending="additive",vertex_colors=fix_vertex_colors,shading='smooth')
viewer.add_surface((mov_vertices,mov_face),blending="additive",vertex_colors=mov_vertex_colors,shading='smooth')

<Surface layer 'Surface [1]' at 0x7f6f60beb5b0>

In [6]:
mov_mesh.export('/home/tmurakami/src/flow_analysis/human_analysis/01_output/220715_prefrontal_q2_R01_wm_refined.ply')
fix_mesh.export('/home/tmurakami/src/flow_analysis/human_analysis/01_output/220715_prefrontal_q2_R01_pia_refined.ply')

b'ply\nformat binary_little_endian 1.0\ncomment https://github.com/mikedh/trimesh\nelement vertex 685\nproperty float x\nproperty float y\nproperty float z\nelement face 1268\nproperty list uchar int vertex_indices\nend_header\nu\xc3\x91D\xf0\xadyD\x10\xc4\xaaCU\x06\xa1D\xecQ\x80D\x9f\xdexCjy\xa3Dk\xd3wDp\xea\xd1C2\x81\x8fD\xa1\x89\x82D\xc6\x0b\x10C\xd4 \x8fD\x98\x01sD;\x1e\xf8C\x01rzD\x98\x95}D\x8d\x83\xd0C\xec\xe5\x82D.\xa3\x80D\r"\x86C\x82WxD\x92\x17\xa2D+3\x8cEe\xabND\x90\xec\x9cD\t\xaf\x8aEY\xeesDfO\x9cD\xd3c\x86E\xfcX\x8dD\x8f\x03\xa4D\xe54\x89E\xf5(\x12E\x9c\x92\x99D\xa6\xf6\x10EC~\x1aE\x95\x94\x9cD\xdb\x80\tEj\x90\x19Ex\x02\xa1DB\xe5\x15E\t\xab\x0bE\x91R\x98D\x9bT\x19E^\xd9\xb4D\xe5\xdf\x80DV\xd9\x8cC\x81\xbf\xa8DC#\x85D\xfc:\xdcB1\xf7\xa4D\xfc\xf2\x9dD{X}E\x9bR\xb5Dz\x97\xa3D\xe8\x99\x81E!\x86\xa1D\xbeg\xa2D\xdd\x04\x84Ee\xde\xa1D\x17\xa8\xa8D,v\x89E\xf3\xcb!E\x94\r\xa7D\xc9\xf0\x11E8F#Ei2\xa2D\xf7\x02\x05E\xf2\t\x1eE\xb9\xc5\x98D\x05\x1b\xf5Dbc\x15E\xa0_\x95D\x91\xea\xffDT\x1